In [ ]:
# ----------------------------------------------------
# 🚀 CÓDIGO COMPLETO EN UNA SOLA CELDA DE COLAB 🚀
# ----------------------------------------------------

# --- I. INSTALACIÓN Y CONFIGURACIÓN ---
print("1. Instalando PySpark y dependencias...")
!pip install pyspark findspark
print("------------------------------------------")

import findspark
findspark.init()

import os
import json
import time
import random
import shutil
from datetime import datetime

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, current_timestamp
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

# Directorio de datos (Simula el Tópico de Kafka)
OUTPUT_DIR = "/content/streaming_data"

if os.path.exists(OUTPUT_DIR):
    # Limpia el directorio de ejecuciones previas
    shutil.rmtree(OUTPUT_DIR)

os.makedirs(OUTPUT_DIR)
print(f"Directorio de 'tópico' creado en: {OUTPUT_DIR}")


# --- II. FUNCIÓN GENERADORA (PRODUCTOR SIMULADO) ---

def generate_and_save_data(num_files=30, records_per_file=10, delay_seconds=0.5):
    """
    Escribe archivos JSON en el directorio de salida en un hilo separado.
    """
    print(f"\n2. Generador de datos (Productor Simulado) iniciando... (Se ejecutará en segundo plano)")
   
    for file_index in range(num_files):
        data_to_write = []
        for record_index in range(records_per_file):
            record = {
                'id': (file_index * records_per_file) + record_index,
                'timestamp': datetime.now().isoformat(),
                'sensor_id': random.choice(['S101', 'S102', 'S103']),
                'pressure_psi': random.randint(50, 150),
                'alert_level': random.choice(['LOW', 'NORMAL', 'HIGH'])
            }
            data_to_write.append(record)

        filename = f"batch_{file_index}_{int(time.time())}.json"
        filepath = os.path.join(OUTPUT_DIR, filename)

        # Escribir JSON por línea (formato JSONLines)
        with open(filepath, 'w') as f:
            for record in data_to_write:
                f.write(json.dumps(record) + '\n')

        # print(f"  -> Lote {file_index+1}/{num_files} creado.") # Silenciar la salida constante
        time.sleep(delay_seconds)
           
    print("\n   => Generador de datos ha terminado de escribir archivos.")

# Usamos threading para que la generación de datos no bloquee el inicio de PySpark
from threading import Thread
producer_thread = Thread(target=generate_and_save_data, args=(30, 10, 1.5))
producer_thread.start()


# --- III. CONFIGURACIÓN E INICIO DE SPARK ---
print("\n3. Configurando Spark Session...")
spark = SparkSession.builder\
    .appName("ColabStreamingSingleCell")\
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

# Definir el esquema de los datos
data_schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("timestamp", StringType(), True),
    StructField("sensor_id", StringType(), True),
    StructField("pressure_psi", IntegerType(), True),
    StructField("alert_level", StringType(), True)
])

# Leer el directorio en modo streaming
streaming_df = spark \
    .readStream \
    .schema(data_schema) \
    .option("maxFilesPerTrigger", 1) \
    .json(OUTPUT_DIR)

print("   => Streaming DataFrame configurado. Iniciando transformación...")

# --- IV. TRANSFORMACIÓN Y LANZAMIENTO DE LA CONSULTA ---

# Tarea de ejemplo: Seleccionar solo alertas 'HIGH' y añadir timestamp de procesamiento.
output_df = streaming_df \
    .filter(col("alert_level") == "HIGH") \
    .withColumn("processed_at", current_timestamp()) \
    .select(
        "sensor_id",
        "pressure_psi",
        "alert_level",
        "processed_at"
    )

# Iniciar la consulta de streaming
query = output_df \
    .writeStream \
    .outputMode("append") \
    .format("console") \
    .option("truncate", "false") \
    .trigger(processingTime='5 seconds') \
    .start()

print("\n4. Procesamiento de Streaming Iniciado. La salida se mostrará cada 5 segundos.")
print("   Presiona el botón de 'Stop' o detiene la ejecución del notebook para finalizar.")

# Mantener la celda viva hasta que el productor haya terminado de escribir todos los archivos.
# Esto asegura que PySpark tenga tiempo de procesar todo el flujo.
producer_thread.join()

print("\n5. El Generador ha terminado. PySpark continuará procesando cualquier dato restante.")

# Opcional: Para detener el streaming después de que el productor termine.
# for stream in spark.streams.active:
#     stream.stop()
# spark.stop()